# 02 — PCA experiments

E0 raw-pixel baseline, E1 PCA + 3 classifiers, E3 eigenspectrum

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

from src.config import OFFSETS
from src.data import load_coil20, make_split
from src.reduction import fit_pca, pca_project

In [ ]:
X, y, pose = load_coil20()

## PCA sanity checks

Run on every split: eigenvalues sorted, eigenvectors orthonormal, N-th eigenvalue ~0, match with sklearn (cross-check only).

In [ ]:
from sklearn.decomposition import PCA  # cross-check only

for off in OFFSETS:
    tr, _ = make_split(pose, off)
    N = len(tr)
    m, V, lam = fit_pca(X[tr])
    assert V.shape == (X.shape[1], N - 1) and lam.shape == (N - 1,)
    assert np.all(np.diff(lam) <= 0)
    assert np.allclose(V.T @ V, np.eye(N - 1), atol=1e-10)
    assert np.linalg.svd(X[tr] - m, compute_uv=False)[-1] ** 2 / N < 1e-20   # dropped component
    assert np.allclose(lam * N / (N - 1), PCA().fit(X[tr]).explained_variance_[:N - 1])
    Z = pca_project(X[tr], m, V, N - 1)
    assert np.allclose(Z.mean(axis=0), 0) and np.allclose(Z.var(axis=0), lam)
print("PCA checks passed")